In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 100000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
# ------------------

torch.manual_seed(1337)

In [5]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    print(f"Length of text: {len(text)} characters")

Length of text: 1115394 characters


In [6]:
 #here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
print(f'{itos}, \n and: {stoi}')
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) 
encode_salut = encode("hello") # decoder: take a list of integers, output a string
print(encode_salut)
print(f' this is decode: {decode(encode_salut)}')

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}, 
 and: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 

In [7]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
print(data)

tensor([18, 47, 56,  ..., 45,  8,  0])


In [8]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    print(f'ix: {ix}')
    print(f'x: {x}')
    print(f'y: {y}')
    return x, y

get_batch('train')

ix: tensor([ 76049, 234249, 934904, 560986, 971401, 579495, 193625, 348340, 406276,
        114168, 953630, 364202, 728777, 417968, 193494, 261116, 357341, 442374,
        202931, 950443, 685221,  91324,  50128, 702412,  62072,  43095, 125128,
        343491, 344273, 851264, 135467, 578465])
x: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54],
        [57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46],
        [43,  1, 51, 39, 63,  1, 40, 43],
        [58, 46, 43,  1, 43, 39, 56, 57],
        [39, 58, 47, 53, 52, 12,  1, 37],
        [53, 56, 43,  1, 21,  1, 41, 39],
        [50, 39, 52, 63,  1, 47, 58, 57],
        [56, 53, 63,  1, 42, 47, 42,  1],
        [39, 51,  1, 39, 44, 56, 39, 47],
        [17, 24, 21, 38, 13, 14, 17, 32],
        [ 1, 39, 52, 42,  1, 45,

(tensor([[24, 43, 58,  5, 57,  1, 46, 43],
         [44, 53, 56,  1, 58, 46, 39, 58],
         [52, 58,  1, 58, 46, 39, 58,  1],
         [25, 17, 27, 10,  0, 21,  1, 54],
         [57, 43, 60, 43, 52,  1, 63, 43],
         [60, 43, 42,  8,  0, 25, 63,  1],
         [56, 42,  5, 57,  1, 57, 39, 49],
         [43, 57, 58, 63,  6,  1, 58, 46],
         [43,  1, 51, 39, 63,  1, 40, 43],
         [58, 46, 43,  1, 43, 39, 56, 57],
         [39, 58, 47, 53, 52, 12,  1, 37],
         [53, 56, 43,  1, 21,  1, 41, 39],
         [50, 39, 52, 63,  1, 47, 58, 57],
         [56, 53, 63,  1, 42, 47, 42,  1],
         [39, 51,  1, 39, 44, 56, 39, 47],
         [17, 24, 21, 38, 13, 14, 17, 32],
         [ 1, 39, 52, 42,  1, 45, 43, 50],
         [ 1, 58, 46, 39, 58,  1, 42, 53],
         [ 1, 61, 53, 59, 50, 42,  1, 21],
         [59, 57, 40, 39, 52, 42,  1, 40],
         [52, 42,  8,  0,  0, 23, 21, 26],
         [45, 53, 42, 57,  0, 23, 43, 43],
         [52,  1, 61, 39, 57,  1, 51, 53],
         [3

In [9]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [13]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [33]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        #print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 99999: train loss 2.4563, val loss 2.4869

r w cl serepstowhengast s rs:
Wht r orge T:
KI's,
Se ty tho and h n, animy hthinietow s
Coneed tather ghorshithy wifot: war,
Haliceeirm den,
Whr, manoupres
buthumor h ses;
tr, wherat iknerveethe m r ased 'verd! ter d dan icecell SBAs, wethe acere bon'ds t d fthackemy
ak O:
Atier' wked cach het, RIn be
Ahingepr, fof mare art t ld llicowawis:
F wime, Gorth ue dofin t tthalilage byorupache.
Whalllo om senove-be sthorofuis hoaso y m.



BREThee st apache
Wheperak reroun, arthen alldicowhouemmy'tatst


In [34]:
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))



H:
TEDYou rrtint k, s, ppay, l my:
We oure t:
Me.
Wind s hie d;
S:
MERY hust,
ORLBudou
TETERere BRCathere s pot tindencrce thak boulr. bomorld t mataryor S:
D at H:
R:
unthe n r Go aloue anonar:

R:
ARUKI is sthe OLIfatcolo ghemotlthat y th harom housour.
D:
id h sir. whaberd d ue and;
Ift t PEENGENour t yo hist hoent bes KINCon t!
D:
Toouns.
o.
PENaine m,

Beal; bros
I n ghos bed: thiu ink t mon juromanoreallfurrinas? y key ber tud RO:

Ton plineiroge; at gr s, s ind.
Too I gsar thre airile.
D
